# Co2L: Evaluación Class-IL

En este notebook evaluamos Co$^2$L en el escenario **Class-Incremental Learning**. 
Comparamos dos enfoques:
1. **NCM (Nearest Class Mean)**: Usando prototipos de clase en el espacio de embeddings.
2. **Sin NCM (Logits agregados)**: Concatenando las salidas de las cabezas lineales de cada tarea.

In [1]:
import torch
import os
from torch.utils.data import ConcatDataset, DataLoader
from models import CNN, Co2LModel
from dataloaders import SequentialCIFAR10
from utils import load_co2l_model
from utils_class_il import compute_class_prototypes, evaluate_ncm, evaluate_multi_head_class_il

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

In [2]:
seq_cifar = SequentialCIFAR10(batch_size=128, buffer_size=500)
NUM_TASKS = 5

for task_id in range(NUM_TASKS):
    ckpt_path = f"checkpoints/co2l/task_{task_id}.pth"
    if not os.path.exists(ckpt_path):
        continue
    
    print(f"\n--- Evaluando Estado tras Tarea {task_id} ---")
    
    # 1. Cargar el modelo de la tarea actual
    model = load_co2l_model(ckpt_path, device)
    
    # 2. Obtener Loader de Test combinado (todas las tareas hasta ahora)
    all_test_datasets = []
    for tid in range(task_id + 1):
        all_test_datasets.append(seq_cifar.get_task_test_dataset(tid))
    
    combined_test_loader = DataLoader(ConcatDataset(all_test_datasets), batch_size=128, shuffle=False)
    
    # --- ENFOQUE 1: NCM (RECALCULADO) ---
    # Recalculamos prototipos con los datasets de entrenamiento completos para ver potencial del backbone
    all_train_datasets = []
    for tid in range(task_id + 1):
        all_train_datasets.append(seq_cifar.get_task_train_dataset(tid, remap_labels=False))
    
    combined_train_loader = DataLoader(ConcatDataset(all_train_datasets), batch_size=128, shuffle=False)
    current_prototypes = compute_class_prototypes(model, combined_train_loader, device)
    
    acc_ncm = evaluate_ncm(model, combined_test_loader, current_prototypes, device)
    print(f"Precisión Class-IL (NCM):           {acc_ncm:.2f}%")
    
    # --- ENFOQUE 2: SIN NCM (LOGITS) ---
    # La función ya imprime el resultado con el formato original del script
    _ = evaluate_multi_head_class_il(model, combined_test_loader, device, seq_cifar.task_classes)